In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
# Define the CNN model
def create_cnn_model(input_shape=(256, 256, 3), num_classes=78):
    model = models.Sequential()

    # Convolutional layers
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Flatten layer
    model.add(layers.Flatten())

    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))  # Dropout for regularization

    model.add(layers.Dense(num_classes, activation='softmax'))

    # Compile the model
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# Specify the paths for training, teasting and validation data
train_data_dir = 'C:\\Users\\Amrutha Y\\Desktop\\Train'
validation_data_dir = 'C:\\Users\\Amrutha Y\\Desktop\\Validation'
test_data_dir = 'C:\\Users\\Amrutha Y\\Desktop\\Test'
# Image data augmentation to improve model generalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

# Set batch size
batch_size = 10

# Load and augment training data
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(256, 256),
    batch_size=batch_size,
    class_mode='categorical')

# Load validation data
validation_generator = test_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(256, 256),
    batch_size=batch_size,
    class_mode='categorical')
# Load and augment test data
test_generator = test_datagen.flow_from_directory(
    test_data_dir,  # Specify the path to your test data
    target_size=(256, 256),
    batch_size=batch_size,
    class_mode='categorical')

# Define a ModelCheckpoint callback
checkpoint = ModelCheckpoint('model_checkpoint.h5', 
                             save_best_only=True, 
                             monitor='val_loss', 
                             mode='min', 
                             verbose=1)

# Create the CNN model
cnn_model = create_cnn_model()

# Train the model
history = cnn_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 16,
    epochs=50,  # Adjust the number of epochs as needed
    callbacks=[checkpoint],
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // 16)
# Evaluate the model on the test data
test_loss, test_accuracy = cnn_model.evaluate(test_generator)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss}")
# Save the trained model
actual_model=cnn_model.save('plant_prediction_model.h5')


Found 1950 images belonging to 78 classes.
Found 546 images belonging to 78 classes.
Found 468 images belonging to 78 classes.
Epoch 1/50
 98/121 [=======================>......] - ETA: 49s - loss: 4.4287 - accuracy: 0.0138WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 6050 batches). You may need to use the repeat() function when building your dataset.

Epoch 1: val_loss improved from inf to 4.35672, saving model to model_checkpoint.h5
24/24 [==============================] - 30s 1s/step - loss: 4.3567 - accuracy: 0.0128
Test Accuracy: 1.28%
Test Loss: 4.356717586517334
